# Expected Wait

In [1]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_rt_headways,
    get_schedule_feed_entries,
)

### Parameters

In [2]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

{'TARGET_DATES': ['2026-06-15',
  '2026-06-16',
  '2026-06-17',
  '2026-06-18',
  '2026-06-19'],
 'TARGET_ROUTE_SHORT_NAME': 'ECR',
 'TARGET_SCHEDULE_FEED': 'Bay Area 511 SamTrans Schedule'}

In [3]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

### Implementation

In [4]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [5]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

,service_date,feed_key
0,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e
1,2026-06-19,d71492afe126cd61f629b17c3c056fcd
2,2026-06-18,d71492afe126cd61f629b17c3c056fcd
3,2026-06-17,d71492afe126cd61f629b17c3c056fcd
5,2026-06-16,d71492afe126cd61f629b17c3c056fcd


,feed_key,last_seen_date,gtfs_dataset_name,shape_id,route_id,route_short_name,ct,geometry
0,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."
1,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
2,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
3,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."


In [6]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

,service_date,feed_key,base64_url,_valid_from
0,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-05-16 03:00:00.573378+00:00
1,2026-06-19,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
2,2026-06-18,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
3,2026-06-17,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
4,2026-06-16,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00


In [7]:
# Get the realtime headway between every pair of consecutive trips, at every
# stop on the pattern rather than only the timepoints
schedule_rt_headways = get_schedule_rt_headways(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
schedule_rt_headways


,service_date,first_trip_key,second_trip_key,stop_sequence,stop_id,shape_id,first_actual_departure,second_actual_arrival,rt_headway_seconds,first_scheduled_departure_sec,second_scheduled_arrival_sec,scheduled_headway_seconds
0,2026-06-15,c8c8031789732fd986518ec203acde2a,f9b18b7a5eef9cd86713db5af47a749e,1,332014,ECR2068,2026-06-15 18:15:00,2026-06-15 18:27:00,720,65400,66300,900
1,2026-06-15,f9b18b7a5eef9cd86713db5af47a749e,3712792ffc168bd236a9488e43012b6c,1,332014,ECR2068,2026-06-15 18:27:00,2026-06-15 19:46:00,4740,66300,71100,4800
2,2026-06-15,3712792ffc168bd236a9488e43012b6c,4433aa6310aa729627cad33b98110fbd,1,332014,ECR2068,2026-06-15 19:46:00,2026-06-15 22:18:00,9120,71100,80100,9000
3,2026-06-15,7f2779002a08bac8c4013694a5d7c267,c8c8031789732fd986518ec203acde2a,2,332170,ECR2068,2026-06-15 17:43:55,2026-06-15 18:16:42,1967,63775,65562,1787
4,2026-06-15,c8c8031789732fd986518ec203acde2a,f9b18b7a5eef9cd86713db5af47a749e,2,332170,ECR2068,2026-06-15 18:16:42,2026-06-15 18:30:42,840,65562,66462,900
...,...,...,...,...,...,...,...,...,...,...,...,...
34696,2026-06-18,3ca0efd2eb313a529d99c3ad3a619f65,ebe776fef9731d85a9c9d8fe2b271972,75,332019,ECR2069,2026-06-18 21:24:00,2026-06-18 21:31:00,420,76740,77640,900
34697,2026-06-18,ebe776fef9731d85a9c9d8fe2b271972,2e4cd72b3bf47e83ccb6605c8b96165d,75,332019,ECR2069,2026-06-18 21:31:00,2026-06-18 21:44:00,780,77640,78540,900
34698,2026-06-18,2e4cd72b3bf47e83ccb6605c8b96165d,9c88df7a0b055a24ec223400d24eeeaa,75,332019,ECR2069,2026-06-18 21:44:00,2026-06-18 22:21:00,2220,78540,80340,1800
34699,2026-06-18,9c88df7a0b055a24ec223400d24eeeaa,2af105a17ad2ed813e009a5d60cf9d54,75,332019,ECR2069,2026-06-18 22:21:00,2026-06-18 23:10:00,2940,80340,83220,2880


In [8]:
schedule_rt_headways.loc[(schedule_rt_headways.stop_sequence == 2) & (schedule_rt_headways.first_actual_departure.dt.hour == 14)].sort_values(by="first_actual_departure")

,service_date,first_trip_key,second_trip_key,stop_sequence,stop_id,shape_id,first_actual_departure,second_actual_arrival,rt_headway_seconds,first_scheduled_departure_sec,second_scheduled_arrival_sec,scheduled_headway_seconds
8560,2026-06-16,1e5e64de925a318351c2ca22f534c487,e34222ef5927fb0ea09a182e0181c0de,2,347016,ECR2069,2026-06-16 14:07:21,2026-06-16 14:23:21,960,50781,51681,900
3161,2026-06-16,592c12750d35312527fc3b24c6eb5694,1fe05bd25f052f83a02b99127e86ee49,2,332170,ECR2068,2026-06-16 14:12:42,2026-06-16 14:27:42,900,51162,52062,900
8561,2026-06-16,e34222ef5927fb0ea09a182e0181c0de,dddc5e163fe6acb05376e9137349eb2c,2,347016,ECR2069,2026-06-16 14:23:21,2026-06-16 14:37:21,840,51681,52581,900
3162,2026-06-16,1fe05bd25f052f83a02b99127e86ee49,c76467417f84e7e72b842608c9257e06,2,332170,ECR2068,2026-06-16 14:27:42,2026-06-16 14:45:42,1080,52062,52962,900
8562,2026-06-16,dddc5e163fe6acb05376e9137349eb2c,bd059e6ebbd84c7b02033980a53914b3,2,347016,ECR2069,2026-06-16 14:37:21,2026-06-16 14:53:21,960,52581,53481,900
3163,2026-06-16,c76467417f84e7e72b842608c9257e06,b7830e7ab18ef2fbf2199cb19167fbb2,2,332170,ECR2068,2026-06-16 14:45:42,2026-06-16 14:58:42,780,52962,53862,900
8563,2026-06-16,bd059e6ebbd84c7b02033980a53914b3,a76ebc8439dee9d5ba13699b2915e327,2,347016,ECR2069,2026-06-16 14:53:21,2026-06-16 15:07:34,853,53481,54394,913
3164,2026-06-16,b7830e7ab18ef2fbf2199cb19167fbb2,9ab6143152446fc3d84b7010677cf763,2,332170,ECR2068,2026-06-16 14:58:42,2026-06-16 15:13:55,913,53862,54775,913
19081,2026-06-17,e3e056faa6260ff79221bfb352de1088,7a600b471ec440bf0faec14788563da2,2,347016,ECR2069,2026-06-17 14:07:21,2026-06-17 14:21:21,840,50781,51681,900
13587,2026-06-17,d098ec90e4b145c804aea86ed7ca08b5,f5fc3de894390b6ef733eeded2d980ab,2,332170,ECR2068,2026-06-17 14:13:42,2026-06-17 14:42:42,1740,51162,52962,1800


In [9]:
# The query carries the headway itself, so the distribution is ready to look at
schedule_rt_headways["rt_headway_seconds"].describe()

count    34701.000000
mean      1006.716089
std        597.948576
min          0.000000
25%        720.000000
50%        960.000000
75%       1212.000000
max      18060.000000
Name: rt_headway_seconds, dtype: float64

### Results

In [10]:
# Headways per stop, pooled over every selected date
headway_summary = (
    schedule_rt_headways
    .groupby(["shape_id", "stop_sequence", "stop_id"])
    .agg(
        n_headways=("rt_headway_seconds", "size"),
        mean_headway_seconds=("rt_headway_seconds", "mean"),
        median_headway_seconds=("rt_headway_seconds", "median"),
        max_headway_seconds=("rt_headway_seconds", "max"),
        headway_variance=("rt_headway_seconds", "var"),
    )
    .reset_index()
)
# a passenger arriving at random waits half the mean headway only when service is
# even; the variance term is the penalty for irregular service
headway_summary["expected_wait_seconds"] = (
    headway_summary["mean_headway_seconds"] / 2
    * (
        1
        + headway_summary["headway_variance"]
        / headway_summary["mean_headway_seconds"] ** 2
    )
)
headway_summary["excess_wait_seconds"] = (
    headway_summary["expected_wait_seconds"]
    - headway_summary["mean_headway_seconds"] / 2
)
headway_summary

,shape_id,stop_sequence,stop_id,n_headways,mean_headway_seconds,median_headway_seconds,max_headway_seconds,headway_variance,expected_wait_seconds,excess_wait_seconds
0,ECR2068,1,332014,67,3187.164179,1860.0,18060,1.088122e+07,3300.620073,1707.037983
1,ECR2068,2,332170,199,1152.783920,960.0,3826,3.188387e+05,714.682692,138.290732
2,ECR2068,3,332232,212,1085.844340,960.0,3060,1.975799e+05,633.902030,90.979860
3,ECR2068,4,332238,217,1061.225806,960.0,3060,1.853166e+05,617.925446,87.312543
4,ECR2068,5,332215,220,1046.236364,960.0,3060,1.830273e+05,610.587577,87.469395
...,...,...,...,...,...,...,...,...,...,...
151,ECR2069,71,332214,225,981.208889,900.0,3600,3.769450e+05,682.686403,192.081958
152,ECR2069,72,332220,222,973.900901,900.0,3720,3.898665e+05,687.107598,200.157147
153,ECR2069,73,332216,222,974.936937,900.0,3780,3.885295e+05,686.727230,199.258761
154,ECR2069,74,332171,223,971.618834,900.0,3780,3.865329e+05,684.721194,198.911777


In [11]:
# Total headways observed per day
# note fewer headways on start and end of the week
schedule_rt_headways["first_actual_departure"].dt.date.value_counts()

first_actual_departure
2026-06-17    10662
2026-06-18    10508
2026-06-16    10430
2026-06-15     3101
Name: count, dtype: int64

In [12]:
# The longest gaps in service, which are what riders actually notice
(
    schedule_rt_headways
    .sort_values("rt_headway_seconds", ascending=False)
    .head(20)
    [["shape_id", "stop_sequence", "stop_id", "first_actual_departure", "second_actual_arrival", "rt_headway_seconds"]]
)

,shape_id,stop_sequence,stop_id,first_actual_departure,second_actual_arrival,rt_headway_seconds
3117,ECR2068,1,332014,2026-06-16 11:11:00,2026-06-16 16:12:00,18060
29681,ECR2069,1,347609,2026-06-18 05:07:00,2026-06-18 10:06:00,17940
13547,ECR2068,1,332014,2026-06-17 18:12:00,2026-06-17 21:53:00,13260
29684,ECR2069,1,347609,2026-06-18 15:36:00,2026-06-18 19:07:00,12660
24213,ECR2068,1,332014,2026-06-18 18:11:00,2026-06-18 21:18:00,11220
8534,ECR2069,1,347609,2026-06-16 19:25:00,2026-06-16 22:23:00,10680
19050,ECR2069,1,347609,2026-06-17 16:36:00,2026-06-17 19:25:00,10140
19045,ECR2069,1,347609,2026-06-17 09:51:00,2026-06-17 12:37:00,9960
13541,ECR2068,1,332014,2026-06-17 11:26:00,2026-06-17 14:11:00,9900
29683,ECR2069,1,347609,2026-06-18 12:51:00,2026-06-18 15:36:00,9900
